In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, log_loss, accuracy_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV
import warnings
warnings.filterwarnings("ignore")

########################
## 1) LOAD DETAILED DATA
########################

men_reg = pd.read_csv("data/MRegularSeasonDetailedResults.csv")
women_reg = pd.read_csv("data/WRegularSeasonDetailedResults.csv")
men_tour = pd.read_csv("data/MNCAATourneyDetailedResults.csv")
women_tour = pd.read_csv("data/WNCAATourneyDetailedResults.csv")

# Keep data from 2010 onward, including partial 2025 if present
men_reg = men_reg[men_reg["Season"] >= 2010]
women_reg = women_reg[women_reg["Season"] >= 2010]
men_tour = men_tour[men_tour["Season"] >= 2010]
women_tour = women_tour[women_tour["Season"] >= 2010]

print("Men Regular Season shape:", men_reg.shape)
print("Women Regular Season shape:", women_reg.shape)
print("Men Tourney shape:", men_tour.shape)
print("Women Tourney shape:", women_tour.shape)

############################
## 2) COMBINE & ENGINEER STATS
############################

# Combine men & women regular season
reg_combined = pd.concat([men_reg, women_reg], ignore_index=True)

# Prepare winner/loser perspective data
winner_cols = reg_combined.copy()
loser_cols = reg_combined.copy()

# Create columns from winner perspective
winner_cols["TeamID"] = winner_cols["WTeamID"]
winner_cols["OpponentID"] = winner_cols["LTeamID"]
winner_cols["TeamScore"] = winner_cols["WScore"]
winner_cols["OpponentScore"] = winner_cols["LScore"]
winner_cols["TeamLoc"] = winner_cols["WLoc"]
winner_cols["Win"] = 1

# For loser perspective
loser_cols["TeamID"] = loser_cols["LTeamID"]
loser_cols["OpponentID"] = loser_cols["WTeamID"]
loser_cols["TeamScore"] = loser_cols["LScore"]
loser_cols["OpponentScore"] = loser_cols["WScore"]

# Map location (if winner was Home, loser is Away, etc.)
loc_map = {"H": "A", "A": "H", "N": "N"}
loser_cols["TeamLoc"] = loser_cols["WLoc"].map(loc_map)
loser_cols["Win"] = 0

# Select columns of interest
team_game_stats = pd.DataFrame()

# Basic subset from reg_combined for each perspective
basic_cols = [
    "Season", "TeamID", "OpponentID", "TeamScore", "OpponentScore",
    "TeamLoc", "Win",
    "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA",
    "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF",
    "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA",
    "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF"
]

winner_cols = winner_cols[basic_cols]
loser_cols = loser_cols[basic_cols]

team_game_stats = pd.concat([winner_cols, loser_cols], ignore_index=True)

# Aggregation: group by (Season, TeamID)
# We'll sum up these advanced stats, then compute means & rates
grouped = team_game_stats.groupby(["Season", "TeamID"])

def possessions(fga, off_reb, to, fta):
    return fga - off_reb + to + 0.44*fta

# Build aggregated DF
df_agg = grouped.agg(
    GamesPlayed=("TeamID", "size"),
    TotalPoints=("TeamScore","sum"),
    OppPoints=("OpponentScore","sum"),
    Wins=("Win","sum"),

    FGM_sum=("WFGM","sum"),  # careful usage, but we have WFGM for 'Team' perspective
    FGA_sum=("WFGA","sum"),
    FGM3_sum=("WFGM3","sum"),
    FGA3_sum=("WFGA3","sum"),
    FTM_sum=("WFTM","sum"),
    FTA_sum=("WFTA","sum"),
    OR_sum=("WOR","sum"),
    DR_sum=("WDR","sum"),
    TO_sum=("WTO","sum"),

    Opp_FGM_sum=("LFGM","sum"),
    Opp_FGA_sum=("LFGA","sum"),
    Opp_FTM_sum=("LFTM","sum"),
    Opp_FTA_sum=("LFTA","sum"),
    Opp_OR_sum=("LOR","sum"),
    Opp_DR_sum=("LDR","sum"),
    Opp_TO_sum=("LTO","sum"),
).reset_index()

# Compute derived stats
df_agg["WinPct"] = df_agg["Wins"] / df_agg["GamesPlayed"]
df_agg["AvgPointsFor"] = df_agg["TotalPoints"] / df_agg["GamesPlayed"]
df_agg["AvgPointsAgainst"] = df_agg["OppPoints"] / df_agg["GamesPlayed"]
df_agg["ScoreMargin"] = df_agg["AvgPointsFor"] - df_agg["AvgPointsAgainst"]

df_agg["FGP"] = df_agg["FGM_sum"] / df_agg["FGA_sum"].replace(0, np.nan)
df_agg["FG3P"] = df_agg["FGM3_sum"] / df_agg["FGA3_sum"].replace(0, np.nan)
df_agg["FTP"] = df_agg["FTM_sum"] / df_agg["FTA_sum"].replace(0, np.nan)

df_agg["AvgOR"] = df_agg["OR_sum"] / df_agg["GamesPlayed"]
df_agg["AvgDR"] = df_agg["DR_sum"] / df_agg["GamesPlayed"]
df_agg["AvgTO"] = df_agg["TO_sum"] / df_agg["GamesPlayed"]

# Offensive & Defensive possessions
df_agg["OffPoss"] = (
    df_agg["FGA_sum"]
    - df_agg["OR_sum"]
    + df_agg["TO_sum"]
    + 0.44*df_agg["FTA_sum"]
)
df_agg["DefPoss"] = (
    df_agg["Opp_FGA_sum"]
    - df_agg["Opp_OR_sum"]
    + df_agg["Opp_TO_sum"]
    + 0.44*df_agg["Opp_FTA_sum"]
)

df_agg["OffEff"] = (df_agg["TotalPoints"] / df_agg["OffPoss"].replace(0, np.nan))*100
df_agg["DefEff"] = (df_agg["OppPoints"] / df_agg["DefPoss"].replace(0, np.nan))*100

# Replace inf & NaN
df_agg = df_agg.replace([np.inf, -np.inf], np.nan).fillna(0)

# Now df_agg has advanced stats for each (Season, TeamID), including 2025 if present
stats_key = df_agg.set_index(["Season","TeamID"])

###############################
## 3) SPLIT TOURNAMENT: TRAIN & VAL
###############################
men_tour = men_tour[men_tour["Season"] <= 2024]  # up to 2024
women_tour = women_tour[women_tour["Season"] <= 2024]

tourney_combined = pd.concat([men_tour, women_tour], ignore_index=True)

# We'll train on 2010-2023, validate on 2024
train_games = tourney_combined[tourney_combined["Season"] < 2024].copy()
val_games = tourney_combined[tourney_combined["Season"] == 2024].copy()

# Create Team1ID, Team2ID, and label
def prep_matches(df):
    df = df.copy()
    df["Team1ID"] = df[["WTeamID","LTeamID"]].min(axis=1)
    df["Team2ID"] = df[["WTeamID","LTeamID"]].max(axis=1)
    df["Result"] = (df["Team1ID"] == df["WTeamID"]).astype(int)
    return df[["Season","Team1ID","Team2ID","Result","WLoc"]]

train_matches = prep_matches(train_games)
val_matches = prep_matches(val_games)

# For team1 location encoding
def encode_loc(row):
    """
    Convert winner-based WLoc into Team1's perspective:
    row["WLoc"] is winner's location:
      'H' if winner was home, 'A' if winner was away, 'N' if neutral.
    row["Result"] == 1 means Team1 was winner, 0 means Team1 was loser.
    """
    mapping = {"H":1, "A":-1, "N":0}

    loc_winner = row["WLoc"]  # winner's location
    if loc_winner == "N":
        return 0

    if row["Result"] == 1:
        # Team1 is winner => Team1's location is row["WLoc"]
        return mapping[loc_winner]
    else:
        # Team1 is loser => we flip home/away
        if loc_winner in ["H","A"]:
            return -mapping[loc_winner]
        return 0


train_matches["Team1Loc"] = train_matches.apply(encode_loc, axis=1)
val_matches["Team1Loc"] = val_matches.apply(encode_loc, axis=1)

###############################
## 4) MERGE STATS FOR TRAIN
###############################
feature_cols = [
    "WinPct","AvgPointsFor","AvgPointsAgainst","ScoreMargin","FGP","FG3P","FTP",
    "AvgOR","AvgDR","AvgTO","OffEff","DefEff"
]

def merge_features(match_df):
    m = match_df.join(
        stats_key, on=["Season","Team1ID"], rsuffix="_T1"
    ).join(
        stats_key, on=["Season","Team2ID"], rsuffix="_T2"
    )
    # Drop rows missing stats
    m.dropna(inplace=True)
    # Build diff columns
    for col in feature_cols:
        m[f"{col}_diff"] = m[col] - m[f"{col}_T2"]
    # final feature set
    feats = [f"{c}_diff" for c in feature_cols] + ["Team1Loc"]
    X = m[feats]
    y = m["Result"]
    return m, X, y

train_merged, X_train, y_train = merge_features(train_matches)
val_merged, X_val, y_val = merge_features(val_matches)
print("Train shape:", X_train.shape, "Val shape:", X_val.shape)

###########################
## 5) TRAIN & VALIDATE MODEL
###########################
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, log_loss

rf_clf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators':[100,200,300],
    'max_depth':[5,10,None],
    'min_samples_leaf':[1,3,5],
    'min_samples_split':[2,5,10],
}

scorer = make_scorer(log_loss, greater_is_better=False, needs_proba=True)

search = RandomizedSearchCV(
    rf_clf, param_distributions=param_grid,
    n_iter=10, scoring=scorer, cv=5,
    random_state=42, n_jobs=-1
)
search.fit(X_train, y_train)

best_rf = search.best_estimator_
print("Best Params:", search.best_params_)

# Evaluate on validation
val_probs = best_rf.predict_proba(X_val)[:,1]
from sklearn.metrics import brier_score_loss, accuracy_score, roc_auc_score

brier = brier_score_loss(y_val, val_probs)
ll = log_loss(y_val, val_probs)
acc = accuracy_score(y_val, (val_probs>=0.5).astype(int))
auc = roc_auc_score(y_val, val_probs)

print(f"Val Brier={brier:.3f}, LogLoss={ll:.3f}, ACC={acc:.3f}, AUC={auc:.3f}")

#############################
## 6) GENERATE 2025 PREDICTIONS
#############################
# SampleSubmissionStage2 includes 2025 matchups
submission_df = pd.read_csv("data/SampleSubmissionStage2.csv")
submission_df[["Season","Team1ID","Team2ID"]] = submission_df["ID"].str.split("_", expand=True)
submission_df["Season"] = submission_df["Season"].astype(int)
submission_df["Team1ID"] = submission_df["Team1ID"].astype(int)
submission_df["Team2ID"] = submission_df["Team2ID"].astype(int)

# Merge team stats for 2025 from df_agg for each team
# We replicate the match/diff logic. We'll assume neutral site => Team1Loc=0
sub_merged = submission_df.join(
    stats_key, on=["Season","Team1ID"], rsuffix="_T1"
).join(
    stats_key, on=["Season","Team2ID"], rsuffix="_T2"
)

# For each of the feature_cols, build diff
for col in feature_cols:
    sub_merged[f"{col}_diff"] = sub_merged[col] - sub_merged[f"{col}_T2"]

sub_merged["Team1Loc"] = 0  # Typically neutral in the tournament

# handle any missing stats
sub_merged.fillna(0, inplace=True)  # or median if you prefer

pred_feats = [f"{c}_diff" for c in feature_cols] + ["Team1Loc"]
pred_probs_2025 = best_rf.predict_proba(sub_merged[pred_feats])[:,1]

submission_df["Pred"] = pred_probs_2025
submission_df[["ID","Pred"]].to_csv("final_submission_2025.csv", index=False)
print("✅ final_submission_2025.csv generated!")


Men Regular Season shape: (83674, 34)
Women Regular Season shape: (80626, 34)
Men Tourney shape: (934, 34)
Women Tourney shape: (894, 34)
Train shape: (1694, 13) Val shape: (134, 13)
Best Params: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 10}
Val Brier=0.199, LogLoss=0.578, ACC=0.672, AUC=0.753
✅ final_submission_2025.csv generated!
